# Sequential Architecture — Google ADK

**Pattern:** Sequential (A → B → C pipeline)  
**Framework:** Google ADK  
**Task:** Research city data → Summarize → Format travel report

## ADK Sequential Pattern

```
User Query
     │
     ▼
┌─────────────────────────────────────────────────┐
│         sequential_travel_reporter              │
│                                                 │
│  STEP 1: fetch_city_data(city) × N cities       │
│               │                                 │
│  STEP 2: Write 2-sentence summaries (LLM)       │
│               │                                 │
│  STEP 3: Format into markdown report (LLM)      │
│                                                 │
└─────────────────────────────────────────────────┘
     │
     ▼
  Final Report
```

**ADK approach:** Sequential pipeline is encoded in the **instruction** as numbered steps.  
The single agent follows the steps deterministically.  
No separate agents or graph nodes — instruction engineering does the work.

## Setup

In [ ]:
import asyncio
import os
from dotenv import load_dotenv
from google.adk.agents import Agent
from google.adk.runners import InMemoryRunner
from google.adk.sessions import InMemorySessionService
from google.genai import types as genai_types

load_dotenv()
assert os.getenv("GOOGLE_API_KEY"), "Set GOOGLE_API_KEY in .env"
print("✓ Ready")

## Tools

In [ ]:
def fetch_city_data(city: str) -> dict:
    """Fetch raw travel data for a city.
    Args:
        city: City name.
    Returns:
        Dict with weather, safety, time, and highlights.
    """
    data = {
        "tokyo":     {"weather": "Clear, 18°C", "safety": "Low", "time": "22:30 JST", "highlights": "Shibuya, Senso-ji"},
        "paris":     {"weather": "Partly Cloudy, 16°C", "safety": "Low", "time": "15:30 CET", "highlights": "Eiffel Tower, Louvre"},
        "bangalore": {"weather": "Rainy, 25°C", "safety": "Medium", "time": "20:00 IST", "highlights": "Lalbagh, Nandi Hills"},
    }
    key = city.lower()
    return {"city": city, **data[key]} if key in data else {"error": f"No data for '{city}'."}

# Test it
print(fetch_city_data("Tokyo"))

## Agent with Sequential Instruction

In [ ]:
agent = Agent(
    name="sequential_travel_reporter",
    model="gemini-2.0-flash",
    description="Produces a formatted travel report via a strict Research → Summarize → Format pipeline.",
    instruction="""You are a travel report generator. For any city comparison request, follow these steps EXACTLY:

STEP 1 — RESEARCH: Call fetch_city_data() for EACH city. Never skip a city.

STEP 2 — SUMMARIZE: For each city, write a concise 2-sentence summary using the fetched data.
  Format: "[City]: [2 sentences covering weather, safety, key attraction]"

STEP 3 — FORMAT: Compile all summaries into a polished travel report:
  - Use '## Travel Report' as the top header
  - Use '### [City Name]' as each city's section header
  - Include weather, safety level, local time, and top attraction for each city
  - End with a '## Recommendation' section naming the best city

Complete all 3 steps before responding.""",
    tools=[fetch_city_data],
)

print(f"Agent: {agent.name} | {len(agent.tools)} tool(s)")

## Run with Streaming

In [ ]:
async def run_sequential(cities: list) -> str:
    session_service = InMemorySessionService()
    session = await session_service.create_session(
        app_name="sequential_travel_reporter", user_id="user_01"
    )
    runner = InMemoryRunner(agent=agent, session_service=session_service)
    query = f"Create a travel report comparing: {', '.join(cities)}."

    print(f"Query: {query}\n")
    print("--- Pipeline trace (STEP 1: tool calls) ---")

    final_response = ""
    async for event in runner.run_async(
        user_id=session.user_id,
        session_id=session.id,
        new_message=genai_types.Content(role="user", parts=[genai_types.Part(text=query)]),
    ):
        if hasattr(event, "tool_call") and event.tool_call:
            print(f"  [STEP 1] fetch_city_data({event.tool_call.args})")
        elif event.is_final_response() and event.content:
            for part in event.content.parts:
                if part.text:
                    final_response += part.text

    return final_response


report = await run_sequential(["Tokyo", "Paris", "Bangalore"])
print("\n--- Final Report (STEPS 2 + 3: LLM output) ---")
print(report)

## Key Takeaways

| What You Saw | Why It Matters |
|---|---|
| Numbered instruction steps | ADK encodes pipeline order in the prompt — no graph nodes |
| Single agent with 1 tool | Simpler than multi-agent setups when a single LLM can follow the steps |
| Tool calls stream first, then LLM output | Confirms Step 1 (tools) completed before Steps 2+3 (LLM) |

### Sequential: All 4 Frameworks Compared
| Framework | How Order is Enforced | Data Passing |
|---|---|---|
| LangChain | Python function call order | Variables |
| LangGraph | `add_edge(A, B)` graph edges | TypedDict state |
| CrewAI | `Process.sequential` + `context=` | Task context injection |
| ADK | Numbered instruction steps | LLM context window |

### Next: [02-Parallel →](../../02-Parallel/ADK/parallel.ipynb)